In [1]:
%%capture
%run "./multiple-linear-regression.ipynb"
# Run the prerequisite notebook - capture no outputs here

In [2]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import cross_val_score

# 1. Update the Split (if you want to strictly follow your new snippet)
# Note: We include y_cat for stratification because y (numerical) cannot be stratified directly
X_train, X_test, y_train, y_test, cat_train, cat_test = train_test_split(
    X, y, y_cat, test_size=0.2, random_state=42, stratify=y_cat
)

# ... [Keep your Scaling and Model Training code here] ...

# 2. Predictions and Binning
y_pred_num = model.predict(X_test_scaled)
y_pred_category = [bin_aqi(val) for val in y_pred_num]

# 3. Accuracy and Classification Report
# We compare the predicted categories against the actual categories (cat_test)
print("Accuracy:", accuracy_score(cat_test, y_pred_category))
print(classification_report(cat_test, y_pred_category))

# 4. Macro F1 Score
macro_f1 = f1_score(cat_test, y_pred_category, average='macro')
print("Macro F1:", macro_f1)

# 5. Cross-Validation
# Since cross_val_score needs a single estimator, we calculate it on the regression 
# R^2 score by default, or you can specify 'neg_mean_squared_error'
scores = cross_val_score(model, X_train_scaled, y_train, cv=5) 

print("Cross-val R^2 Mean:", scores.mean())
print("Std Dev:", scores.std())

# 6. Training vs Test Performance (R^2 Score for Regression)
print("Train R^2 Score:", model.score(X_train_scaled, y_train))
print("Test R^2 Score:", model.score(X_test_scaled, y_test))

Accuracy: 0.9934590812290842
              precision    recall  f1-score   support

        Good       1.00      0.98      0.99       836
    Moderate       0.99      1.00      0.99      2287
        Poor       1.00      1.00      1.00         2
Satisfactory       1.00      0.99      0.99      3449

    accuracy                           0.99      6574
   macro avg       1.00      0.99      0.99      6574
weighted avg       0.99      0.99      0.99      6574

Macro F1: 0.9944585219886444
Cross-val R^2 Mean: 0.9999180169675425
Std Dev: 1.225656617166649e-06
Train R^2 Score: 0.9999153519917647
Test R^2 Score: 0.9999128086991811


In [3]:
def get_aqi_report(pm25, pm10, no2, so2, co, o3, season):

    # Input validation
    inputs = [pm25, pm10, no2, so2, co, o3]
    if any(n < 0 for n in inputs):
        return "Error: Pollutant levels cannot be negative."

    # Build input row matching training columns
    input_row = {col: 0.0 for col in X.columns}
    input_row.update({
        'pm2.5': pm25,
        'pm10': pm10,
        'no2': no2,
        'so2': so2,
        'co': co,
        'o3': o3
    })

    # Robust season mapping (fixes Post-Monsoon/case issues)
    season_key = str(season).strip().lower()
    season_to_col = {
        'winter': 'season_Winter',
        'summer': 'season_Summer',
        'post-monsoon': 'season_Post-Monsoon',
        'post monsoon': 'season_Post-Monsoon',
        'monsoon': None  # baseline when drop_first=True
    }

    if season_key not in season_to_col:
        valid = ", ".join(['Winter', 'Summer', 'Monsoon', 'Post-Monsoon'])
        return f"Error: Invalid season '{season}'. Use one of: {valid}."

    season_col = season_to_col[season_key]
    if season_col is not None:
        input_row[season_col] = 1.0

    # Scale and predict
    input_df = pd.DataFrame([input_row])[X.columns]
    input_scaled = scaler.transform(input_df)
    prediction = model.predict(input_scaled)[0]
    prediction = model.predict(input_scaled)[0]

    if prediction <= 50:
        cat, advice = "Good", "Minimal impact"
    elif prediction <= 100:
        cat, advice = "Satisfactory", "Minor breathing discomfort to sensitive people"
    elif prediction <= 200:
        cat, advice = "Moderate", "Breathing discomfort to people with lungs/heart disease"
    elif prediction <= 300:
        cat, advice = "Poor", "Breathing discomfort to most people on prolonged exposure"
    else:
        cat, advice = "Severe", "Affects healthy people and seriously impacts those with existing diseases"

    # 6. Output
    print(f"{'='*40}")
    print(f"  AQI PREDICTION REPORT: {str(season).upper()}")
    print(f"{'='*40}")
    print(f"  Predicted AQI: {prediction:.2f}")
    print(f"  Category:      {cat}")
    print(f"  Health Advice: {advice}")
    print(f"{'='*40}")

    return prediction, cat

In [4]:
def test_cli():
    """
    CLI loop for manual AQI prediction input.
    """
    season_options = {
        "1": "Winter",
        "2": "Summer",
        "3": "Monsoon",
        "4": "Post-Monsoon",
    }

    print("AQI Prediction CLI")
    print("-" * 40)
    print("Enter pollutant values in these units:")
    print("PM2.5, PM10, NO2, SO2, O3 -> µg/m³")
    print("CO -> mg/m³")

    while True:
        print("\nChoose an option:")
        print("1) Enter pollutant parameters")
        print("2) Exit")

        choice = input("Enter choice (1/2): ").strip()

        if choice == "1":
            try:
                pm25 = float(input("PM2.5 (µg/m³): ").strip())
                pm10 = float(input("PM10 (µg/m³): ").strip())
                no2 = float(input("NO2 (µg/m³): ").strip())
                so2 = float(input("SO2 (µg/m³): ").strip())
                co = float(input("CO (mg/m³): ").strip())
                o3 = float(input("O3 (µg/m³): ").strip())

                print("\nSelect season:")
                for k, v in season_options.items():
                    print(f"{k}) {v}")
                season_input = input("Enter season number (1-4) or name: ").strip()

                season = season_options.get(season_input, season_input)

                result = get_aqi_report(pm25, pm10, no2, so2, co, o3, season)
                if isinstance(result, str):
                    print(result)

            except ValueError:
                print("Invalid input. Please enter numeric values for pollutants.")

        elif choice == "2":
            print("Exiting CLI.")
            break

        else:
            print("Invalid choice. Enter 1 or 2.")


# Start CLI
test_cli()


AQI Prediction CLI
----------------------------------------
Enter pollutant values in these units:
PM2.5, PM10, NO2, SO2, O3 -> µg/m³
CO -> mg/m³

Choose an option:
1) Enter pollutant parameters
2) Exit

Select season:
1) Winter
2) Summer
3) Monsoon
4) Post-Monsoon
  AQI PREDICTION REPORT: WINTER
  Predicted AQI: 179.22
  Category:      Moderate
  Health Advice: Breathing discomfort to people with lungs/heart disease

Choose an option:
1) Enter pollutant parameters
2) Exit
Exiting CLI.
